<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #00137cff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

#  **DS105W Mini-Project 1: London Air Quality Analysis (Winter Term 2025/2026)**

## **_Data Transformation Notebook_**
- 👤 Name: Laurie Taylor
- 📛 Candidate Number: 73691
- 📅 Date: 21st February 2026
- 🎯 Purpose: Investigate Weekday vs Weekend Air Pollution Patterns Using OpenWeather API **_- Refine Datasets and Load as CSV file._**

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #c6a8ffff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

### ⚙️ **Importing additional libraries:**

In [42]:
import json
import os
import pandas as pd 
import holidays

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #c6a8ffff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

### 💭 Personal Reflection:
 
 I imported holidays to give a record of UK public holidays by year, so I can isolate the days that behave like weekends but aren't and exclude them from both groups to keep the weekday vs weekend comparison accurate.

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #00137cff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

## **Step 1: Load JSON & Parse to DataFrame**

In [43]:
locations = ["marylebone_road", "richmond"]
dfs = []

for location in locations:
    with open(f"../data/air_pollution_{location}.json", "r") as f:
        raw = json.load(f)

    # flattening list of hourly observationss
    df = pd.json_normalize(raw["list"]).rename(columns={"dt": "timestamp"})

    # only keeping components.* and timestamp
    component_cols = [c for c in df.columns if c.startswith("components.")]
    df = df[["timestamp"] + component_cols]

    # renaming components.* for plain names
    df.columns = ["timestamp"] + [c.replace("components.", "") for c in component_cols]

    df["location"] = location
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

# tz-aware London time
df_all["timestamp"] = (
    pd.to_datetime(df_all["timestamp"], unit="s", utc=True)
      .dt.tz_convert("Europe/London")
)

print(df_all.head())
print("Hourly rows:", len(df_all))



                  timestamp      co     no    no2    o3   so2  pm2_5   pm10  \
0 2020-11-27 00:00:00+00:00  347.14  33.53  41.13  0.01  7.51  18.81  21.35   
1 2020-11-27 01:00:00+00:00  293.73  11.18  42.16  0.21  7.27  15.68  18.17   
2 2020-11-27 02:00:00+00:00  277.04   5.64  41.81  0.32  7.33  15.31  17.65   
3 2020-11-27 03:00:00+00:00  277.04   4.75  41.13  0.40  7.57  15.78  18.02   
4 2020-11-27 04:00:00+00:00  277.04   4.47  40.44  0.43  7.87  16.73  18.96   

    nh3         location  
0  0.25  marylebone_road  
1  0.01  marylebone_road  
2  0.01  marylebone_road  
3  0.02  marylebone_road  
4  0.03  marylebone_road  
Hourly rows: 90652


<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #00137cff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

## **Step 2: Filtering Time Horizon to Whole Years**

**Reason for Decision**:
- The API data begins 27/11/2020, meaning 2020 is an incomplete year. To make analysis easier, I'm filtering to 01/01/2021 - 31/12/2025 which gives me 5 full calendar years (>200 weekends). 
- With regards to COVID and restricted activity potentially skewing my results, I'll wait and see how the data reflects this in Data Analysis.

In [33]:
df_all = df_all[df_all["timestamp"].dt.year.between(2021, 2025)]

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #00137cff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

## **Step 3: Selecting Pollutants**

**Reason for Decision:**

- I selected the pollutants by deciding what I wanted to test in Data Analysis. I was most interested in the effects of traffic emissions, picking up signs of secondary atmospheric pollution, and making sure to keep the pollutants that are most relevant to public health concerns. The 3 chosen pollutants are all listed on the DEFRA DAQI.

- Therefore, I decided to keep:
    - **NO₂:** Suitable indicator of traffic emissions; interested to know whether or not weekend reductions in traffic activity are reflected in NO₂ decreases.
    - **PM₂.₅:** Public health relevance, varied source profile (traffic emissions, residential heating etc.); can explore whether weekend effects extend beyond traffic pollutants.
    - **O₃ :** Weekend ozone effect exploration (reductions in. NO emissions may reduce ozone titration and result in higher weekend ozone concentrations).

- SO₂ and PM₁₀ are also listed on the DAQI, so they're both 'regulated' in the UK, however were omitted to maintain data clarity. PM₁₀ often mirrors PM₂.₅, so it wouldn't provide additional insights. SO₂ levels are generally low since London's shift away from coal usage.

- CO, NO and NH₃ are not listed on the DAQI, therefore would make it more difficult to categorise in Transformation and Analysis. CO was excluded because the London levels consistently lie within health limits. NO and NH₃ were omitted because NO is a precursor for NO₂, and NO₂ is associated with agricultural emissions (therefore less relevant to urban weeday-weekend pattern detection).

In [44]:
keep = ["timestamp", "location", "no2", "pm2_5", "o3"]
df_all = df_all[keep].rename(columns={"pm2_5": "pm25"})

print(df_all.head())
print("Hourly rows (2021–2025):", len(df_all))

                  timestamp         location    no2   pm25    o3
0 2020-11-27 00:00:00+00:00  marylebone_road  41.13  18.81  0.01
1 2020-11-27 01:00:00+00:00  marylebone_road  42.16  15.68  0.21
2 2020-11-27 02:00:00+00:00  marylebone_road  41.81  15.31  0.32
3 2020-11-27 03:00:00+00:00  marylebone_road  41.13  15.78  0.40
4 2020-11-27 04:00:00+00:00  marylebone_road  40.44  16.73  0.43
Hourly rows (2021–2025): 90652


<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #00137cff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

## **Step 4: Computing Daily Means (Maintaining DAQI-consistency)**

(Reason for choosing Daily Means as Temporal Aggregation Outlined in **NB01 Introduction**)

**Reasons for Choosing the DEFRA DAQI:**
- The OpenWeather API page suggests that for UK AQIs (and a select few other regions), users should use the resources on the 'Air Quality Index Levels Scale' page. I followed the page, and it presented the DEFRA DAQI for each of my chosen pollutants.¹⁰
- The DEFRA Daily Air Quality Index (DAQI) will help more accurately translate pollution levels into a standardised health-based scale thats used in official UK communications.¹¹
- It has a 1-10 index, with 4 categorising bands: Low, Moderate, High and Very High. Because it's specific to the UK, the calibration will be more accurate - giving more resolution at levels relevant to London.
- Finally, using DAQI means that my findings can be directly compared with DEFRA's own statistics in the Analysis.

- Each pollutant uses a different averaging time:
    - NO₂: hourly mean
    - PM₂.₅: daily mean
    - O₃: running 8-hour mean
- Therefore, as a final stage in my data cleaning, I need to compute these correctl from the hourly series first. 

In [ ]:
hourly = df_all.copy()

# grouping hours of same calendar days
hourly["date"] = hourly["timestamp"].dt.floor("D")

# PM2.5 daily mean
pm25_daily = (
    hourly.groupby(["location", "date"], as_index=False)["pm25"]
    .mean()
    .rename(columns={"pm25": "pm25_daily_mean"})
)

# NO2 daily maximum hourly value 
no2_daily = (
    hourly.groupby(["location", "date"], as_index=False)["no2"]
    .max()
    .rename(columns={"no2": "no2_daily_max_1h"})
)

# O3 8-hour rolling mean, then daily max of that series
hourly = hourly.sort_values(["location", "timestamp"])

hourly["o3_8h_mean"] = (
    hourly.groupby("location")["o3"]
    .rolling(window=8, min_periods=8)
    .mean()
    .reset_index(level=0, drop=True)
)

o3_daily = (
    hourly.groupby(["location", "date"], as_index=False)["o3_8h_mean"]
    .max()
    .rename(columns={"o3_8h_mean": "o3_daily_max_8h"})
)

# sanity check (available hourly values per day)
counts = (
    hourly.groupby(["location", "date"], as_index=False)[["no2", "pm25", "o3"]]
    .count()
    .rename(columns={"no2": "no2_n", "pm25": "pm25_n", "o3": "o3_n"})
)

# combining into one daily dataset
daily_metrics = (
    pm25_daily
    .merge(no2_daily, on=["location", "date"], how="inner")
    .merge(o3_daily,  on=["location", "date"], how="inner")
    .merge(counts,    on=["location", "date"], how="left")
)

print(daily_metrics.head())
print("Daily rows (raw):", len(daily_metrics))
print(daily_metrics[["no2_n", "pm25_n", "o3_n"]].describe())

          location                      date  pm25_daily_mean  \
0  marylebone_road 2020-11-27 00:00:00+00:00        15.890417   
1  marylebone_road 2020-11-28 00:00:00+00:00        26.863333   
2  marylebone_road 2020-11-29 00:00:00+00:00        30.290417   
3  marylebone_road 2020-11-30 00:00:00+00:00        12.165417   
4  marylebone_road 2020-12-01 00:00:00+00:00         2.435833   

   no2_daily_max_1h  o3_daily_max_8h  no2_n  pm25_n  o3_n  
0             45.24          4.49000     24      24    24  
1             49.35          5.41125     24      24    24  
2             44.55         10.64125     24      24    24  
3             43.18         48.68250     24      24    24  
4             39.07         58.20625     24      24    24  
Daily rows (raw): 3816
             no2_n       pm25_n         o3_n
count  3816.000000  3816.000000  3816.000000
mean     23.755765    23.755765    23.755765
std       2.268502     2.268502     2.268502
min       1.000000     1.000000     1.000000
2

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #00137cff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

## **Step 5: Day-of-Week Tagging (Excluding Public Holidays)**

**Weekend Definition:**
- I've defined a weekend as Saturday and Sunday only. This is so that my research is comparable to most official data. I will exclude public holidays from the refined datasetsto avoid compressing the weekday-weekend differential. If the data later shows that Fridays should be included, I will adjust my decision with reasoning.

- Minimum hourly data threshold of 75% is commonly adopted in UK air quality reporting.¹² To ensure robustness, a stricter threshold of 20 hours (≈83% coverage) was adopted.

In [49]:
# weekends
daily_metrics["is_weekend"] = daily_metrics["date"].dt.dayofweek >= 5
daily_metrics["day_name"] = daily_metrics["date"].dt.day_name()

# england public hols
uk_holidays = holidays.UnitedKingdom(years=range(2021, 2026), subdiv="ENG")
holiday_dates = set(uk_holidays.keys())  # datetime.date objects
daily_metrics["is_public_holiday"] = daily_metrics["date"].dt.date.isin(holiday_dates)

# excluding public hols
daily_no_hols = daily_metrics.loc[~daily_metrics["is_public_holiday"]].copy()

# excluding days with poor hourly coverage (less than 20 hours)
before_coverage = len(daily_no_hols)

daily_final = daily_no_hols[
    (daily_no_hols["no2_n"] >= 20) &
    (daily_no_hols["pm25_n"] >= 20) &
    (daily_no_hols["o3_n"] >= 20)
].copy()

after_coverage = len(daily_final)
poor_coverage_excluded = before_coverage - after_coverage

print(daily_final["day_name"].value_counts().sort_index())
print(f"Public holidays excluded: {daily_metrics['is_public_holiday'].sum()}")
print(f"Poor hourly coverage excluded: {poor_coverage_excluded}")
print(f"Final clean daily rows: {len(daily_final):,}")
print(f"Total excluded: {len(daily_metrics) - len(daily_final)}")

day_name
Friday       528
Monday       490
Saturday     534
Sunday       532
Thursday     530
Tuesday      536
Wednesday    532
Name: count, dtype: int64
Public holidays excluded: 96
Poor hourly coverage excluded: 38
Final clean daily rows: 3,682
Total excluded: 134


<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #00137cff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

## **Step 6: Save as CSV**

In [50]:
os.makedirs("../data", exist_ok=True)

out_path = "../data/air_pollution_daily_clean.csv"
daily_final.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Rows: {len(daily_final):,}")
print("Columns:", daily_final.columns.tolist())

Saved: ../data/air_pollution_daily_clean.csv
Rows: 3,682
Columns: ['location', 'date', 'pm25_daily_mean', 'no2_daily_max_1h', 'o3_daily_max_8h', 'no2_n', 'pm25_n', 'o3_n', 'is_weekend', 'day_name', 'is_public_holiday']


<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #c6a8ffff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

### 💭 **Personal Reflection:**
- During Data Transformation, I noticed that the calulated weekday/weekend mean and percentage reduction in NO₂ concentration were identical for Marylebone Road and Bloomsbury. To check this wasn't just an error in my previous code I ran the following sanity check, as 2 distinct locations should not have returned identical values across 5-year dataset averages.
- To do this, I compared the NO₂ time series for both locations using .equals() and a difference count. The result confirmed that the entire series was identical across all days. Richmond showed different values in the sanity checks, so I didn't include a test to verify Marylebone and Richmond were distinct.

Test 1:
![Test for Identical Datapoints](../figures/Test.png)

Richmond Showing Distinct Values for NO₂ concentration (Step 8 Sanity Checks):
![Proof that Richmond was Distinct](../figures/Richmond%20Different%20Values.png)

- This indicated that Marylebone Road and Bloomsbury's coordinates were mapped to the same OpenWeather model grid cell. The API provides gridded model data rather than the kind of pinpoint accuracy I needed, which I should have checked before I started Data Collection. I repeated this test with 5 pollutants, to make sure that the duplications I was experiencing were structural and not specific to NO₂. This confirmed my suspicions, and strongly suggested that both locations belonged to the same grid cell:

Test 2:
![Test 2](../figures/Test%202.png)

- As a result, Bloomsbury was omitted from the study to avoid duplication. The final comparison has been changes to Marylebone Road (traffic-dominated) and Richmond (background site), which have distinct time series.

- This process has improved my spatial awareness when working with API datasets and has stressed to me that validating independence between study locations before conducting Data Collection is an important step. I've also learnt that trying to test API grid cell resolution is quite a tricky task, and visualising it is even tricker. I attempted this in NB01 Task 2.2 as an experiment to test my skills in response to these challenges.

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #dbd0f0ff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:800px;color:#212121;">

**Sources:**

¹⁰ OpenWeather - Air Pollution Index Levels, Available at: https://openweathermap.org/air-pollution-index-levels?collection=environmental

¹¹ GOV.UK - Pollutant concentrations for the Daily Air Quality Index (DAQI), December 2025. Available at: https://www.gov.uk/government/publications/health-effects-of-air-pollution/pollutant-concentrations-for-the-daily-air-quality-index-daqi

¹² DEFRA - UK Air Information Resource: Frequently Asked Questions (Data Capture and Validity Criteria). Available at: https://uk-air.defra.gov.uk/air-pollution/faq?question=23&utm


